In [1]:
import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt

from tb_macro.constants import AGE_STRATA, ISO3, START_TIME, END_TIME, SOLVER_KWARGS
from tb_macro.epi import get_base_model, add_flows_to_model, initialise_pops
from tb_macro.inputs import load_demography, load_fertility, load_who_outcomes
from tb_macro.demography import prepare_pop_data_for_entries
from tb_macro.parameters import BASE_PARAMS
from tb_macro.outputs import (
    get_share_folder_file_path,
    get_age_inc,
    get_age_prev,
    get_age_latent,
    get_age_notifs,
    get_age_deaths,
    get_total_pop,
    get_posterior_samples,
    collate_output_table,
)
from tb_macro.plotting import plot_outputs

plt.style.use("ggplot")
pd.options.plotting.backend = "matplotlib"

/Users/jamestrauer/dev/tb_macroeconomics/.pixi/envs/default/lib/python3.13/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


In [2]:
# Model construction
group_popsize, death_rates, age_weights = load_demography(ISO3)
fert_padded = load_fertility(ISO3)
tsr, death_in_unsucc, who_mort = load_who_outcomes(ISO3)
epi_model, disease_state, age_strat, clin_strat, infect_strat = get_base_model(START_TIME, END_TIME)
start_apops = [1000.0] * len(AGE_STRATA) # Arbitrary starting values, inflows determine growth
entry_times, entry_rates = prepare_pop_data_for_entries(group_popsize, START_TIME, sum(start_apops))
add_flows_to_model(
    epi_model, 
    disease_state,
    age_strat,
    clin_strat,
    infect_strat,
    age_weights,
    group_popsize,
    fert_padded,
    death_rates,
    tsr,
    death_in_unsucc,
    entry_times,
    entry_rates,
)
initialise_pops(epi_model, disease_state, age_strat, start_apops)

In [3]:
idata = az.from_netcdf("nuts_100_100_idata.nc")
az.summary(idata)

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
contact_rate,5.888,0.112,5.672,6.088,0.006,0.005,303.0,288.0,1.01
detect_val_2,0.649,0.044,0.560,0.700,0.003,0.002,136.0,167.0,1.02


In [4]:
# Load an idata that was prepared earlier
idata = az.from_netcdf("nuts_100_100_idata.nc")

In [5]:
# Get samples
samples = get_posterior_samples(idata, 5)

In [6]:
# Collate outputs
scen_params = [{}, {"detect_gap_reduction": 0.5}]
sample_labels = []
indicator_funcs = {
    "incidence": get_age_inc,
    "prevalence": get_age_prev,
    "latent": get_age_latent,
    "notifications": get_age_notifs,
    "deaths": get_age_deaths,
    "total_pop": get_total_pop,
}
outputs = [{out: [] for out in indicator_funcs} for _ in scen_params]

for i in range(samples.sizes["sample"]):
    run = f"chain_{int(samples['chain'][i])}/draw_{int(samples['draw'][i])}"
    sample_labels.append(run)
    c_params = {k: float(samples[k].isel(sample=i)) for k in idata.posterior.data_vars}
    for s, s_params in enumerate(scen_params):
        results = epi_model.run(BASE_PARAMS | c_params | s_params, solver_kwargs=SOLVER_KWARGS)
        for out, func in indicator_funcs.items():
            output = func(results, age_strat, disease_state).to_pandas_df()
            output.columns.name = "age_group"
            outputs[s][out].append(output)

W0722 17:40:13.811489 3367950 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


In [7]:
from tb_macro.inputs import get_country_pop, get_single_age_pop_from_ungroups

In [40]:
def assign_age_groups(pops, breaks, name):
    break_ints = [int(a) for a in breaks]
    bins = break_ints + [np.inf]
    pops[name] = pd.cut(pops["Age"], bins=bins, right=False, labels=break_ints)


pop_data = get_country_pop(ISO3)
single_age_pops = get_single_age_pop_from_ungroups(pop_data)
example_out = outputs[0]["incidence"][0]
pops = single_age_pops.copy()
out_groups = [0, 15, 50]
m_group_name = "model_age_group"
o_group_name = "output_age_group"
assign_age_groups(pops, example_out.columns, m_group_name)
assign_age_groups(pops, out_groups, o_group_name)
overlaps = pops.groupby(["Time", m_group_name, o_group_name])["Pop"].sum().reset_index()
model_totals = pops.groupby(["Time", m_group_name])["Pop"].sum().reset_index(name="model_pop")

overlaps.head(50)

,Time,model_age_group,output_age_group,Pop
0,1950,0,0,1953739.8
1,1950,3,0,1302493.2
2,1950,5,0,2431057.0
3,1950,10,0,2393198.0
4,1950,15,15,1509524.4
5,1950,18,15,8674001.6
6,1950,40,15,2875691.0
7,1950,40,50,2678697.0
8,1950,65,50,1013228.0
9,1951,0,0,2135256.0


In [10]:
full_out = collate_output_table(outputs, sample_labels)

In [11]:
def sum_df_over_lower_level(df):
    return df.T.groupby(level=0).sum().T

s_plot = 0
total_pop = sum_df_over_lower_level(full_out[s_plot]["total_pop"])
incs = sum_df_over_lower_level(full_out[s_plot]["incidence"])
notifs = sum_df_over_lower_level(full_out[s_plot]["notifications"])
prevs = sum_df_over_lower_level(full_out[s_plot]["prevalence"])
tb_deaths = sum_df_over_lower_level(full_out[s_plot]["deaths"])
latent = sum_df_over_lower_level(full_out[s_plot]["latent"])

In [12]:
fig = plot_outputs(prevs, incs, notifs, NOTIF_TARGET, tb_deaths, who_mort, latent, None, total_pop, 1980.0, 2050.0, "count")

NameError: name 'NOTIF_TARGET' is not defined

In [ ]:
fig = plot_outputs(prevs, incs, notifs, NOTIF_TARGET, tb_deaths, None, latent, LATENT_TARGET, total_pop, 1980.0, 2050.0, "rate")

In [ ]:
# james_work_gdrive = "/Users/jtrauer/Library/CloudStorage/GoogleDrive-james.trauer@monash.edu/"
# out_path = get_share_folder_file_path(james_work_gdrive) 
# full_out.to_csv(out_path / "full_outputs.csv")